# Benchmark ReLeaf, WeedsGalore and EWI1S with CLIPSeg and Grounded-SAM

This notebook extends the VegAnn benchmark to the three target domains using the **same deterministic 80/10/10 split (seed 42)** used in the thesis experiments.

### Fair benchmark protocol
- Evaluation split: non-empty **validation** images only.
- CLIPSeg: `CIDAS/clipseg-rd64-refined`, prompt **"vegetation"**, threshold **0.5**.
- Grounded-SAM: GroundingDINO + SAM ViT-B, prompt **"plant"**, box threshold **0.25**, text threshold **0.20**.
- Multiple GroundingDINO detections are converted to SAM masks and **unioned**.
- Metrics: image-wise IoU, Dice, Precision, Recall and runtime.
- AgriCLIP→SAM2 reference values are the matched-protocol validation baselines from the final fair comparison.


In [1]:
# CELL 1 — Imports and reproducibility

import os
import sys
import time
import random
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn.functional as F

from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA GeForce RTX 4090


In [2]:
# CELL 2 — Dataset paths

PROJECT_ROOT = Path("/home/zahranadeem/thesis")

RELEAF_ROOT = PROJECT_ROOT / "CropAndWeedAndLeaf"
RELEAF_IMAGES_DIR = RELEAF_ROOT / "images"
RELEAF_LABELS_DIR = RELEAF_ROOT / "labels"

WEEDSGALORE_ROOT = PROJECT_ROOT / "WeedsGalore_dataset"
WEEDS_IMAGE_DIR = WEEDSGALORE_ROOT / "data"
WEEDS_MASK_DIR = WEEDSGALORE_ROOT / "fields" / "ground_truth_semantic"

EWIS1_ROOT = PROJECT_ROOT / "EWIS1"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
MASK_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def find_nested_dataset_dir(root, dirname, extension=".png"):
    candidates = []
    for p in root.rglob(dirname):
        if not p.is_dir():
            continue
        files = [
            x for x in p.iterdir()
            if x.is_file() and x.suffix.lower() == extension.lower()
        ]
        if files:
            candidates.append((len(files), p))
    if not candidates:
        return None
    candidates.sort(reverse=True, key=lambda x: x[0])
    return candidates[0][1]

EWIS1_IMAGE_DIR = find_nested_dataset_dir(EWIS1_ROOT, "images", ".png")
EWIS1_MASK_DIR = find_nested_dataset_dir(EWIS1_ROOT, "masks", ".png")

for name, path in {
    "ReLeaf images": RELEAF_IMAGES_DIR,
    "ReLeaf labels": RELEAF_LABELS_DIR,
    "WeedsGalore images": WEEDS_IMAGE_DIR,
    "WeedsGalore masks": WEEDS_MASK_DIR,
    "EWI1S images": EWIS1_IMAGE_DIR,
    "EWI1S masks": EWIS1_MASK_DIR,
}.items():
    print(f"{name:25s}: {path} | exists={path is not None and Path(path).exists()}")


ReLeaf images            : /home/zahranadeem/thesis/CropAndWeedAndLeaf/images | exists=True
ReLeaf labels            : /home/zahranadeem/thesis/CropAndWeedAndLeaf/labels | exists=True
WeedsGalore images       : /home/zahranadeem/thesis/WeedsGalore_dataset/data | exists=True
WeedsGalore masks        : /home/zahranadeem/thesis/WeedsGalore_dataset/fields/ground_truth_semantic | exists=True
EWI1S images             : /home/zahranadeem/thesis/EWIS1/EWIS1 Maize, Sorghum & Weed Dataset/images/images | exists=True
EWI1S masks              : /home/zahranadeem/thesis/EWIS1/EWIS1 Maize, Sorghum & Weed Dataset/masks/masks | exists=True


In [3]:
# CELL 3 — Common binary-mask helper

def binary_mask_from_array(mask_image):
    arr = np.asarray(mask_image)

    if arr.ndim == 3:
        # Any non-zero RGB pixel is foreground for ordinary binary/semantic masks.
        arr = np.any(arr[..., :3] > 0, axis=2)
    else:
        arr = arr > 0

    return arr.astype(np.uint8)


In [4]:
# CELL 4 — ReLeaf loader (YOLO polygons → binary vegetation mask)

def load_releaf_image_mask(image_path):
    image_path = Path(image_path)
    image = Image.open(image_path).convert("RGB")
    w, h = image.size

    mask = np.zeros((h, w), dtype=np.uint8)
    label_path = RELEAF_LABELS_DIR / f"{image_path.stem}.txt"

    if not label_path.exists():
        return image, mask

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        vals = line.strip().split()
        if len(vals) < 7:
            continue

        coords = [float(v) for v in vals[1:]]
        if len(coords) < 6:
            continue

        pts = []
        for i in range(0, len(coords), 2):
            if i + 1 >= len(coords):
                break
            pts.append([
                int(coords[i] * w),
                int(coords[i + 1] * h)
            ])

        if len(pts) >= 3:
            cv2.fillPoly(
                mask,
                [np.asarray(pts, dtype=np.int32)],
                1
            )

    return image, mask


releaf_pairs = []

for p in sorted(RELEAF_IMAGES_DIR.iterdir()):
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
        lp = RELEAF_LABELS_DIR / f"{p.stem}.txt"
        if lp.exists():
            releaf_pairs.append({
                "image_path": p,
                "label_path": lp,
                "dataset": "ReLeaf"
            })

print("ReLeaf matched pairs:", len(releaf_pairs))


ReLeaf matched pairs: 345


In [5]:
# CELL 5 — WeedsGalore loader

def find_mask_for_stem(mask_dir, stem):
    matches = [
        p for p in mask_dir.glob(f"{stem}.*")
        if p.suffix.lower() in MASK_EXTENSIONS
    ]
    return matches[0] if matches else None


weedsgalore_pairs = []

for p in sorted(WEEDS_IMAGE_DIR.rglob("*")):
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
        mp = find_mask_for_stem(WEEDS_MASK_DIR, p.stem)
        if mp is not None:
            weedsgalore_pairs.append({
                "image_path": p,
                "mask_path": mp,
                "dataset": "WeedsGalore"
            })


def load_weedsgalore_image_mask(item):
    image = Image.open(item["image_path"]).convert("RGB")
    mask = binary_mask_from_array(
        Image.open(item["mask_path"])
    )
    return image, mask


print("WeedsGalore matched pairs:", len(weedsgalore_pairs))


WeedsGalore matched pairs: 156


In [6]:
# CELL 6 — EWI1S loader
# Official weed color used in the existing thesis notebook:
# RGB = (255, 127, 14), alpha > 0

ewis1_pairs = []

for p in sorted(EWIS1_IMAGE_DIR.iterdir()):
    if p.is_file() and p.suffix.lower() == ".png":
        mp = EWIS1_MASK_DIR / p.name
        if mp.exists():
            ewis1_pairs.append({
                "image_path": p,
                "mask_path": mp,
                "dataset": "EWI1S"
            })


def load_ewis1_image_mask(item):
    image = Image.open(item["image_path"]).convert("RGB")

    rgba = np.array(
        Image.open(item["mask_path"]).convert("RGBA")
    )

    rgb = rgba[:, :, :3]
    alpha = rgba[:, :, 3]

    weed = (
        (rgb[:, :, 0] == 255) &
        (rgb[:, :, 1] == 127) &
        (rgb[:, :, 2] == 14) &
        (alpha > 0)
    )

    return image, weed.astype(np.uint8)


print("EWI1S matched pairs:", len(ewis1_pairs))


EWI1S matched pairs: 88


In [7]:
# CELL 7 — Recreate the exact deterministic 80/10/10 splits

TARGET_DATASETS = ["ReLeaf", "WeedsGalore", "EWI1S"]

LOADERS = {
    "ReLeaf": lambda item: load_releaf_image_mask(item["image_path"]),
    "WeedsGalore": load_weedsgalore_image_mask,
    "EWI1S": load_ewis1_image_mask,
}

ALL_PAIRS = {
    "ReLeaf": releaf_pairs,
    "WeedsGalore": weedsgalore_pairs,
    "EWI1S": ewis1_pairs,
}


def split_pairs(
    pairs,
    seed=42,
    train_fraction=0.80,
    val_fraction=0.10
):
    pairs = list(pairs)

    rng = random.Random(seed)
    rng.shuffle(pairs)

    n = len(pairs)
    n_train = int(train_fraction * n)
    n_val = int(val_fraction * n)

    return (
        pairs[:n_train],
        pairs[n_train:n_train + n_val],
        pairs[n_train + n_val:]
    )


def filter_nonempty(pairs, loader):
    good = []
    empty = []

    for item in tqdm(
        pairs,
        desc="Checking masks",
        leave=False
    ):
        _, mask = loader(item)

        if mask.sum() > 0:
            good.append(item)
        else:
            empty.append(item)

    return good, empty


dataset_splits = {}

for dataset_name in TARGET_DATASETS:

    train_records, val_records, test_records = split_pairs(
        ALL_PAIRS[dataset_name],
        seed=SEED
    )

    train_records, empty_train = filter_nonempty(
        train_records,
        LOADERS[dataset_name]
    )

    val_records, empty_val = filter_nonempty(
        val_records,
        LOADERS[dataset_name]
    )

    test_records, empty_test = filter_nonempty(
        test_records,
        LOADERS[dataset_name]
    )

    dataset_splits[dataset_name] = {
        "train": train_records,
        "val": val_records,
        "test": test_records
    }

    print(
        f"{dataset_name:15s} | "
        f"train={len(train_records):4d} | "
        f"val={len(val_records):3d} | "
        f"test={len(test_records):3d} | "
        f"dropped={len(empty_train)+len(empty_val)+len(empty_test)}"
    )

print()
print("Expected validation sizes from the fair comparison:")
print("ReLeaf      : 34")
print("WeedsGalore : 15")
print("EWI1S       : 8")


Checking masks:   0%|          | 0/276 [00:00<?, ?it/s]

Checking masks:   0%|          | 0/34 [00:00<?, ?it/s]

Checking masks:   0%|          | 0/35 [00:00<?, ?it/s]

ReLeaf          | train= 276 | val= 34 | test= 35 | dropped=0


Checking masks:   0%|          | 0/124 [00:00<?, ?it/s]

Checking masks:   0%|          | 0/15 [00:00<?, ?it/s]

... (2 outputs left out)

Checking masks:   0%|          | 0/70 [00:00<?, ?it/s]

Checking masks:   0%|          | 0/8 [00:00<?, ?it/s]

Checking masks:   0%|          | 0/10 [00:00<?, ?it/s]

EWI1S           | train=  70 | val=  8 | test= 10 | dropped=0

Expected validation sizes from the fair comparison:
ReLeaf      : 34
WeedsGalore : 15
EWI1S       : 8


In [8]:
# CELL 8 — Metrics

def compute_binary_metrics(pred_mask, gt_mask):

    pred_mask = np.asarray(pred_mask).astype(bool)
    gt_mask = np.asarray(gt_mask).astype(bool)

    intersection = np.logical_and(
        pred_mask,
        gt_mask
    ).sum()

    union = np.logical_or(
        pred_mask,
        gt_mask
    ).sum()

    pred_pixels = pred_mask.sum()
    gt_pixels = gt_mask.sum()

    iou = (
        intersection / union
        if union > 0
        else (1.0 if gt_pixels == 0 else 0.0)
    )

    denominator = pred_pixels + gt_pixels

    dice = (
        2.0 * intersection / denominator
        if denominator > 0
        else (1.0 if gt_pixels == 0 else 0.0)
    )

    precision = (
        intersection / pred_pixels
        if pred_pixels > 0
        else 0.0
    )

    recall = (
        intersection / gt_pixels
        if gt_pixels > 0
        else 0.0
    )

    return {
        "iou": float(iou),
        "dice": float(dice),
        "precision": float(precision),
        "recall": float(recall)
    }


## Part A — CLIPSeg

This keeps the final VegAnn CLIPSeg setting: **prompt = "vegetation"**, sigmoid threshold = **0.5**.


In [9]:
# CELL 9 — Load CLIPSeg

from transformers import AutoProcessor, CLIPSegForImageSegmentation

CLIPSEG_MODEL_NAME = "CIDAS/clipseg-rd64-refined"
CLIPSEG_PROMPT = "vegetation"
CLIPSEG_THRESHOLD = 0.5

clipseg_processor = AutoProcessor.from_pretrained(
    CLIPSEG_MODEL_NAME
)

clipseg_model = (
    CLIPSegForImageSegmentation
    .from_pretrained(CLIPSEG_MODEL_NAME)
    .to(DEVICE)
)

clipseg_model.eval()

print("CLIPSeg loaded")
print("Prompt:", CLIPSEG_PROMPT)
print("Threshold:", CLIPSEG_THRESHOLD)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIPSeg loaded
Prompt: vegetation
Threshold: 0.5


In [10]:
# CELL 10 — CLIPSeg prediction function

@torch.no_grad()
def clipseg_predict(
    image,
    prompt=CLIPSEG_PROMPT,
    threshold=CLIPSEG_THRESHOLD
):

    inputs = clipseg_processor(
        text=[prompt],
        images=[image],
        padding=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
    }

    outputs = clipseg_model(
        **inputs
    )

    logits = outputs.logits

    if logits.ndim == 3:
        logits = logits.unsqueeze(1)

    logits = F.interpolate(
        logits,
        size=image.size[::-1],
        mode="bilinear",
        align_corners=False
    )

    probability = torch.sigmoid(
        logits
    )[0, 0]

    pred_mask = (
        probability > threshold
    ).cpu().numpy()

    return pred_mask


In [11]:
# CELL 11 — Evaluate CLIPSeg on one target dataset

def evaluate_clipseg_dataset(dataset_name):

    records = dataset_splits[
        dataset_name
    ]["val"]

    loader = LOADERS[
        dataset_name
    ]

    rows = []

    print()
    print("=" * 80)
    print(
        f"CLIPSEG — {dataset_name} VALIDATION "
        f"(n={len(records)})"
    )
    print("=" * 80)

    for idx, item in enumerate(
        tqdm(records, desc=f"CLIPSeg {dataset_name}")
    ):

        image, gt_mask = loader(
            item
        )

        start = time.perf_counter()

        pred_mask = clipseg_predict(
            image
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        runtime = (
            time.perf_counter()
            - start
        )

        metrics = compute_binary_metrics(
            pred_mask,
            gt_mask
        )

        rows.append({
            "Dataset": dataset_name,
            "Method": "CLIPSeg",
            "Prompt": CLIPSEG_PROMPT,
            "index": idx,
            **metrics,
            "runtime_sec": runtime
        })

    per_image_df = pd.DataFrame(
        rows
    )

    summary = {
        "Dataset": dataset_name,
        "Method": "CLIPSeg",
        "Prompt": CLIPSEG_PROMPT,
        "Images": len(per_image_df),
        "IoU_%": 100 * per_image_df["iou"].mean(),
        "Dice_%": 100 * per_image_df["dice"].mean(),
        "Precision_%": 100 * per_image_df["precision"].mean(),
        "Recall_%": 100 * per_image_df["recall"].mean(),
        "Runtime_ms": 1000 * per_image_df["runtime_sec"].mean()
    }

    return (
        pd.DataFrame([summary]),
        per_image_df
    )


In [12]:
# CELL 12 — Run CLIPSeg on all three datasets

clipseg_summaries = []
clipseg_per_image = {}

for dataset_name in TARGET_DATASETS:

    summary_df, per_image_df = (
        evaluate_clipseg_dataset(
            dataset_name
        )
    )

    clipseg_summaries.append(
        summary_df
    )

    clipseg_per_image[
        dataset_name
    ] = per_image_df

    display(
        summary_df.round(3)
    )

clipseg_summary_all = pd.concat(
    clipseg_summaries,
    ignore_index=True
)

display(
    clipseg_summary_all.round(3)
)



CLIPSEG — ReLeaf VALIDATION (n=34)


CLIPSeg ReLeaf:   0%|          | 0/34 [00:00<?, ?it/s]

/home/zahranadeem/thesis/agriclip_sam2/agriclip_env/lib/python3.10/site-packages/transformers/image_processing_utils.py:51: UserWarning: The following named arguments are not valid for `ViTImageProcessor.preprocess` and were ignored: 'padding'
  return self.preprocess(images, **kwargs)


,Dataset,Method,Prompt,Images,IoU_%,Dice_%,Precision_%,Recall_%,Runtime_ms
0,ReLeaf,CLIPSeg,vegetation,34,71.381,82.438,74.521,94.447,15.636



CLIPSEG — WeedsGalore VALIDATION (n=15)


CLIPSeg WeedsGalore:   0%|          | 0/15 [00:00<?, ?it/s]

... (3 outputs left out)

CLIPSeg EWI1S:   0%|          | 0/8 [00:00<?, ?it/s]

/home/zahranadeem/thesis/agriclip_sam2/agriclip_env/lib/python3.10/site-packages/transformers/image_processing_utils.py:51: UserWarning: The following named arguments are not valid for `ViTImageProcessor.preprocess` and were ignored: 'padding'
  return self.preprocess(images, **kwargs)


,Dataset,Method,Prompt,Images,IoU_%,Dice_%,Precision_%,Recall_%,Runtime_ms
0,EWI1S,CLIPSeg,vegetation,8,4.343,7.192,8.041,8.61,178.32


,Dataset,Method,Prompt,Images,IoU_%,Dice_%,Precision_%,Recall_%,Runtime_ms
0,ReLeaf,CLIPSeg,vegetation,34,71.381,82.438,74.521,94.447,15.636
1,WeedsGalore,CLIPSeg,vegetation,15,13.072,21.126,30.989,20.277,9.665
2,EWI1S,CLIPSeg,vegetation,8,4.343,7.192,8.041,8.610,178.320


## Part B — Grounded-SAM

This matches the clean VegAnn benchmark:
- GroundingDINO prompt: **"plant"**
- Box threshold: **0.25**
- Text threshold: **0.20**
- SAM: **ViT-B**
- If GroundingDINO returns multiple boxes, all SAM masks are unioned.
- If no box is detected, prediction is an empty mask.


In [18]:
# CELL 13 — Grounded-SAM configuration
# Uses Hugging Face Grounding DINO Tiny + SAM ViT-B
# This avoids the local GroundingDINO _C CUDA-extension problem.

from pathlib import Path

GROUNDING_DINO_MODEL_NAME = (
    "IDEA-Research/grounding-dino-tiny"
)

SAM_CHECKPOINT = Path(
    "/home/zahranadeem/"
    "Grounded-Segment-Anything/"
    "sam_vit_b_01ec64.pth"
)

GROUNDING_PROMPT = "plant"

BOX_THRESHOLD = 0.25
TEXT_THRESHOLD = 0.20

print("=" * 70)
print("GROUNDED-SAM CONFIGURATION")
print("=" * 70)

print(
    "GroundingDINO:",
    GROUNDING_DINO_MODEL_NAME
)

print(
    "SAM checkpoint:",
    SAM_CHECKPOINT
)

print(
    "SAM exists:",
    SAM_CHECKPOINT.exists()
)

print(
    "Prompt:",
    GROUNDING_PROMPT
)

print(
    "Box threshold:",
    BOX_THRESHOLD
)

print(
    "Text threshold:",
    TEXT_THRESHOLD
)

if not SAM_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"SAM checkpoint not found: "
        f"{SAM_CHECKPOINT}"
    )

GROUNDED-SAM CONFIGURATION
GroundingDINO: IDEA-Research/grounding-dino-tiny
SAM checkpoint: /home/zahranadeem/Grounded-Segment-Anything/sam_vit_b_01ec64.pth
SAM exists: True
Prompt: plant
Box threshold: 0.25
Text threshold: 0.2


In [19]:
# CELL 14 — Load HF Grounding DINO Tiny + SAM ViT-B

import torch

from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection
)

from segment_anything import (
    sam_model_registry,
    SamPredictor
)


# ============================================================
# LOAD GROUNDING DINO
# ============================================================

print(
    "Loading Hugging Face "
    "Grounding DINO Tiny..."
)

grounding_processor = (
    AutoProcessor.from_pretrained(
        GROUNDING_DINO_MODEL_NAME
    )
)

grounding_model = (
    AutoModelForZeroShotObjectDetection
    .from_pretrained(
        GROUNDING_DINO_MODEL_NAME
    )
    .to(DEVICE)
)

grounding_model.eval()

for p in grounding_model.parameters():
    p.requires_grad = False


# ============================================================
# LOAD SAM ViT-B
# ============================================================

print(
    "Loading SAM ViT-B..."
)

sam = sam_model_registry[
    "vit_b"
](
    checkpoint=str(
        SAM_CHECKPOINT
    )
)

sam = sam.to(
    DEVICE
)

sam.eval()

for p in sam.parameters():
    p.requires_grad = False


sam_predictor = SamPredictor(
    sam
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 70)
print("Grounded-SAM loaded successfully")
print("=" * 70)

print(
    "GroundingDINO:",
    GROUNDING_DINO_MODEL_NAME
)

print(
    "SAM:",
    "ViT-B"
)

print(
    "Prompt:",
    GROUNDING_PROMPT
)

print(
    "Box threshold:",
    BOX_THRESHOLD
)

print(
    "Text threshold:",
    TEXT_THRESHOLD
)

Loading Hugging Face Grounding DINO Tiny...
Loading SAM ViT-B...

Grounded-SAM loaded successfully
GroundingDINO: IDEA-Research/grounding-dino-tiny
SAM: ViT-B
Prompt: plant
Box threshold: 0.25
Text threshold: 0.2


In [21]:
# CELL 15 — Grounded-SAM prediction
# HF Grounding DINO → SAM ViT-B
#
# Multiple detections:
#     all SAM masks are UNIONED
#
# No detection:
#     return empty mask
#
# No fake/fallback boxes are used.

@torch.no_grad()
def grounded_sam_predict(
    image,
    prompt=GROUNDING_PROMPT,
    box_threshold=BOX_THRESHOLD,
    text_threshold=TEXT_THRESHOLD
):

    # ========================================================
    # PREPARE IMAGE
    # ========================================================

    image = image.convert(
        "RGB"
    )

    image_np = np.asarray(
        image
    )


    # ========================================================
    # GROUNDING DINO INPUT
    # ========================================================

    # Grounding DINO works more reliably with
    # a period-terminated text query.

    text_prompt = (
        prompt.strip()
    )

    if not text_prompt.endswith(
        "."
    ):
        text_prompt += "."


    inputs = grounding_processor(
        images=image,
        text=text_prompt,
        return_tensors="pt"
    )


    inputs = {
        k: (
            v.to(DEVICE)
            if torch.is_tensor(v)
            else v
        )
        for k, v in inputs.items()
    }


    # ========================================================
    # GROUNDING DINO FORWARD
    # ========================================================

    outputs = grounding_model(
        **inputs
    )


    # ========================================================
    # POST-PROCESS DETECTIONS
    # ========================================================

    target_sizes = torch.tensor(
        [
            [
                image.height,
                image.width
            ]
        ],
        device=DEVICE
    )


    results = (
        grounding_processor
        .post_process_grounded_object_detection(
            outputs,
            inputs["input_ids"],
            threshold=box_threshold,
            text_threshold=text_threshold,
            target_sizes=target_sizes
        )
    )


    result = results[0]


    boxes = result[
        "boxes"
    ]


    # ========================================================
    # NO DETECTION
    # ========================================================

    if boxes is None or len(
        boxes
    ) == 0:

        empty_mask = np.zeros(
            (
                image.height,
                image.width
            ),
            dtype=bool
        )

        return (
            empty_mask,
            0
        )


    # ========================================================
    # CLEAN BOXES
    # ========================================================

    boxes = (
        boxes
        .detach()
        .cpu()
        .numpy()
        .astype(
            np.float32
        )
    )


    valid_boxes = []


    for box in boxes:

        x1, y1, x2, y2 = (
            box.tolist()
        )


        # Clip to image
        x1 = max(
            0.0,
            min(
                x1,
                image.width - 1
            )
        )

        y1 = max(
            0.0,
            min(
                y1,
                image.height - 1
            )
        )

        x2 = max(
            0.0,
            min(
                x2,
                image.width - 1
            )
        )

        y2 = max(
            0.0,
            min(
                y2,
                image.height - 1
            )
        )


        # Remove invalid boxes
        if (
            x2 <= x1
            or y2 <= y1
        ):
            continue


        valid_boxes.append(
            [
                x1,
                y1,
                x2,
                y2
            ]
        )


    # ========================================================
    # NO VALID BOXES AFTER CLEANING
    # ========================================================

    if len(
        valid_boxes
    ) == 0:

        empty_mask = np.zeros(
            (
                image.height,
                image.width
            ),
            dtype=bool
        )

        return (
            empty_mask,
            0
        )


    boxes = np.asarray(
        valid_boxes,
        dtype=np.float32
    )


    num_boxes = len(
        boxes
    )


    # ========================================================
    # SAM IMAGE ENCODING
    # ========================================================

    sam_predictor.set_image(
        image_np
    )


    # ========================================================
    # TRANSFORM GROUNDING-DINO BOXES TO SAM COORDINATES
    # ========================================================

    boxes_torch = torch.as_tensor(
        boxes,
        dtype=torch.float32,
        device=DEVICE
    )


    transformed_boxes = (
        sam_predictor
        .transform
        .apply_boxes_torch(
            boxes_torch,
            image_np.shape[:2]
        )
    )


    # ========================================================
    # SAM PREDICTION
    # ========================================================

    masks, scores, logits = (
        sam_predictor.predict_torch(
            point_coords=None,
            point_labels=None,
            boxes=transformed_boxes,
            multimask_output=False
        )
    )


    # masks:
    # [N, 1, H, W]

    masks = (
        masks[:, 0]
        .detach()
        .cpu()
        .numpy()
        .astype(bool)
    )


    # ========================================================
    # UNION ALL DETECTED PLANT MASKS
    # ========================================================

    pred_mask = np.any(
        masks,
        axis=0
    )


    return (
        pred_mask,
        num_boxes
    )


print(
    "HF Grounded-SAM prediction "
    "function ready."
)

HF Grounded-SAM prediction function ready.


In [22]:
# CELL 15B — Grounded-SAM sanity check

test_record = (
    dataset_splits[
        "ReLeaf"
    ]["val"][0]
)

test_image, test_gt = (
    LOADERS[
        "ReLeaf"
    ](
        test_record
    )
)

test_pred, test_boxes = (
    grounded_sam_predict(
        test_image
    )
)

print(
    "Image size:",
    test_image.size
)

print(
    "GT pixels:",
    int(
        test_gt.sum()
    )
)

print(
    "Predicted pixels:",
    int(
        test_pred.sum()
    )
)

print(
    "GroundingDINO boxes:",
    test_boxes
)

print(
    "IoU:",
    compute_binary_metrics(
        test_pred,
        test_gt
    )["iou"] * 100
)

Image size: (163, 160)
GT pixels: 8745
Predicted pixels: 18351
GroundingDINO boxes: 2
IoU: 43.745358090185675


In [23]:
# CELL 16 — Evaluate Grounded-SAM on one target dataset

def evaluate_grounded_sam_dataset(
    dataset_name
):

    records = dataset_splits[
        dataset_name
    ]["val"]

    loader = LOADERS[
        dataset_name
    ]

    rows = []

    print()
    print("=" * 80)
    print(
        f"GROUNDED-SAM — {dataset_name} VALIDATION "
        f"(n={len(records)})"
    )
    print("=" * 80)

    for idx, item in enumerate(
        tqdm(
            records,
            desc=f"Grounded-SAM {dataset_name}"
        )
    ):

        image, gt_mask = loader(
            item
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        pred_mask, num_boxes = (
            grounded_sam_predict(
                image
            )
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        runtime = (
            time.perf_counter()
            - start
        )

        metrics = compute_binary_metrics(
            pred_mask,
            gt_mask
        )

        rows.append({
            "Dataset": dataset_name,
            "Method": "Grounded-SAM",
            "Prompt": GROUNDING_PROMPT,
            "index": idx,
            **metrics,
            "runtime_sec": runtime,
            "num_boxes": num_boxes
        })

    per_image_df = pd.DataFrame(
        rows
    )

    summary = {
        "Dataset": dataset_name,
        "Method": "Grounded-SAM",
        "Prompt": GROUNDING_PROMPT,
        "Images": len(per_image_df),
        "IoU_%": 100 * per_image_df["iou"].mean(),
        "Dice_%": 100 * per_image_df["dice"].mean(),
        "Precision_%": 100 * per_image_df["precision"].mean(),
        "Recall_%": 100 * per_image_df["recall"].mean(),
        "Runtime_ms": 1000 * per_image_df["runtime_sec"].mean(),
        "Avg_Boxes": per_image_df["num_boxes"].mean(),
        "No_Detection_%": 100 * (
            per_image_df["num_boxes"] == 0
        ).mean()
    }

    return (
        pd.DataFrame([summary]),
        per_image_df
    )


In [24]:
grounded_summaries = []
grounded_per_image = {}

for dataset_name in TARGET_DATASETS:

    summary_df, per_image_df = (
        evaluate_grounded_sam_dataset(
            dataset_name
        )
    )

    grounded_summaries.append(
        summary_df
    )

    grounded_per_image[
        dataset_name
    ] = per_image_df

    display(
        summary_df.round(3)
    )

grounded_summary_all = pd.concat(
    grounded_summaries,
    ignore_index=True
)

display(
    grounded_summary_all.round(3)
)


GROUNDED-SAM — ReLeaf VALIDATION (n=34)


Grounded-SAM ReLeaf:   0%|          | 0/34 [00:00<?, ?it/s]

,Dataset,Method,Prompt,Images,IoU_%,Dice_%,Precision_%,Recall_%,Runtime_ms,Avg_Boxes,No_Detection_%
0,ReLeaf,Grounded-SAM,plant,34,9.467,14.994,11.481,25.732,149.135,1.235,0.0



GROUNDED-SAM — WeedsGalore VALIDATION (n=15)


Grounded-SAM WeedsGalore:   0%|          | 0/15 [00:00<?, ?it/s]

,Dataset,Method,Prompt,Images,IoU_%,Dice_%,Precision_%,Recall_%,Runtime_ms,Avg_Boxes,No_Detection_%
0,WeedsGalore,Grounded-SAM,plant,15,8.213,14.067,23.852,47.855,134.785,1.667,6.667



GROUNDED-SAM — EWI1S VALIDATION (n=8)


Grounded-SAM EWI1S:   0%|          | 0/8 [00:00<?, ?it/s]

,Dataset,Method,Prompt,Images,IoU_%,Dice_%,Precision_%,Recall_%,Runtime_ms,Avg_Boxes,No_Detection_%
0,EWI1S,Grounded-SAM,plant,8,1.505,2.837,2.283,74.811,501.464,1.375,0.0


,Dataset,Method,Prompt,Images,IoU_%,Dice_%,Precision_%,Recall_%,Runtime_ms,Avg_Boxes,No_Detection_%
0,ReLeaf,Grounded-SAM,plant,34,9.467,14.994,11.481,25.732,149.135,1.235,0.000
1,WeedsGalore,Grounded-SAM,plant,15,8.213,14.067,23.852,47.855,134.785,1.667,6.667
2,EWI1S,Grounded-SAM,plant,8,1.505,2.837,2.283,74.811,501.464,1.375,0.000


## Part C — Final thesis-ready comparison

The AgriCLIP→SAM2 values below are the matched-protocol **validation baselines** from the final fair comparison:
- ReLeaf: **84.695%**
- WeedsGalore: **62.195%**
- EWI1S: **17.712%**

Replace/add the exact Dice/Precision/Recall values only if you want those fields in the final cross-method table.


In [25]:
# CELL 18 — AgriCLIP→SAM2 reference results from fair comparison

agriclip_reference = pd.DataFrame([
    {
        "Dataset": "ReLeaf",
        "Method": "AgriCLIP + SAM2",
        "Prompt": "AgriCLIP-generated box",
        "Images": 34,
        "IoU_%": 84.695,
        "Dice_%": 91.439,
        "Precision_%": 87.178,
        "Recall_%": 96.524,
        "Runtime_ms": 26.734
    },
    {
        "Dataset": "WeedsGalore",
        "Method": "AgriCLIP + SAM2",
        "Prompt": "AgriCLIP-generated box",
        "Images": 15,
        "IoU_%": 62.195,
        "Dice_%": 76.350,
        "Precision_%": 76.368,
        "Recall_%": 77.056,
        "Runtime_ms": 28.178
    },
    {
        "Dataset": "EWI1S",
        "Method": "AgriCLIP + SAM2",
        "Prompt": "AgriCLIP-generated box",
        "Images": 8,
        "IoU_%": 17.712,
        "Dice_%": 26.039,
        "Precision_%": 30.231,
        "Recall_%": 27.200,
        "Runtime_ms": 127.032
    }
])

display(
    agriclip_reference.round(3)
)


,Dataset,Method,Prompt,Images,IoU_%,Dice_%,Precision_%,Recall_%,Runtime_ms
0,ReLeaf,AgriCLIP + SAM2,AgriCLIP-generated box,34,84.695,91.439,87.178,96.524,26.734
1,WeedsGalore,AgriCLIP + SAM2,AgriCLIP-generated box,15,62.195,76.350,76.368,77.056,28.178
2,EWI1S,AgriCLIP + SAM2,AgriCLIP-generated box,8,17.712,26.039,30.231,27.200,127.032


In [26]:
# CELL 19 — Final combined benchmark table

clipseg_final = (
    clipseg_summary_all.copy()
)

grounded_final = (
    grounded_summary_all.copy()
)

# Keep common columns for the main comparison.
common_columns = [
    "Dataset",
    "Method",
    "Prompt",
    "Images",
    "IoU_%",
    "Dice_%",
    "Precision_%",
    "Recall_%",
    "Runtime_ms"
]

final_benchmark = pd.concat(
    [
        agriclip_reference[
            common_columns
        ],
        clipseg_final[
            common_columns
        ],
        grounded_final[
            common_columns
        ]
    ],
    ignore_index=True
)

dataset_order = {
    "ReLeaf": 0,
    "WeedsGalore": 1,
    "EWI1S": 2
}

method_order = {
    "AgriCLIP + SAM2": 0,
    "CLIPSeg": 1,
    "Grounded-SAM": 2
}

final_benchmark["_dataset_order"] = (
    final_benchmark["Dataset"]
    .map(dataset_order)
)

final_benchmark["_method_order"] = (
    final_benchmark["Method"]
    .map(method_order)
)

final_benchmark = (
    final_benchmark
    .sort_values(
        [
            "_dataset_order",
            "_method_order"
        ]
    )
    .drop(
        columns=[
            "_dataset_order",
            "_method_order"
        ]
    )
    .reset_index(drop=True)
)

display(
    final_benchmark.style.format({
        "IoU_%": "{:.2f}%",
        "Dice_%": "{:.2f}%",
        "Precision_%": "{:.2f}%",
        "Recall_%": "{:.2f}%",
        "Runtime_ms": "{:.2f}"
    })
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "cross_domain_benchmarks"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

final_benchmark.to_csv(
    OUTPUT_DIR
    / "FINAL_ReLeaf_WeedsGalore_EWI1S_benchmark.csv",
    index=False
)

print(
    "Saved:",
    OUTPUT_DIR
    / "FINAL_ReLeaf_WeedsGalore_EWI1S_benchmark.csv"
)


,Dataset,Method,Prompt,Images,IoU_%,Dice_%,Precision_%,Recall_%,Runtime_ms
0,ReLeaf,AgriCLIP + SAM2,AgriCLIP-generated box,34,84.69%,91.44%,87.18%,96.52%,26.73
1,ReLeaf,CLIPSeg,vegetation,34,71.38%,82.44%,74.52%,94.45%,15.64
2,ReLeaf,Grounded-SAM,plant,34,9.47%,14.99%,11.48%,25.73%,149.14
3,WeedsGalore,AgriCLIP + SAM2,AgriCLIP-generated box,15,62.20%,76.35%,76.37%,77.06%,28.18
4,WeedsGalore,CLIPSeg,vegetation,15,13.07%,21.13%,30.99%,20.28%,9.66
5,WeedsGalore,Grounded-SAM,plant,15,8.21%,14.07%,23.85%,47.86%,134.79
6,EWI1S,AgriCLIP + SAM2,AgriCLIP-generated box,8,17.71%,26.04%,30.23%,27.20%,127.03
7,EWI1S,CLIPSeg,vegetation,8,4.34%,7.19%,8.04%,8.61%,178.32
8,EWI1S,Grounded-SAM,plant,8,1.50%,2.84%,2.28%,74.81%,501.46


Saved: /home/zahranadeem/thesis/cross_domain_benchmarks/FINAL_ReLeaf_WeedsGalore_EWI1S_benchmark.csv


In [27]:
# CELL 20 — Compact IoU table for thesis / presentation

iou_table = (
    final_benchmark
    .pivot(
        index="Dataset",
        columns="Method",
        values="IoU_%"
    )
    .reindex(
        [
            "ReLeaf",
            "WeedsGalore",
            "EWI1S"
        ]
    )
)

display(
    iou_table.style.format(
        "{:.2f}%"
    )
)

iou_table.to_csv(
    OUTPUT_DIR
    / "FINAL_IoU_table_cross_domain.csv"
)


Method,AgriCLIP + SAM2,CLIPSeg,Grounded-SAM
Dataset,,,
ReLeaf,84.69%,71.38%,9.47%
WeedsGalore,62.20%,13.07%,8.21%
EWI1S,17.71%,4.34%,1.50%


In [28]:
# CELL 21 — Save per-image competitor results

for dataset_name in TARGET_DATASETS:

    clipseg_per_image[
        dataset_name
    ].to_csv(
        OUTPUT_DIR
        / f"{dataset_name}_CLIPSeg_per_image.csv",
        index=False
    )

    grounded_per_image[
        dataset_name
    ].to_csv(
        OUTPUT_DIR
        / f"{dataset_name}_GroundedSAM_per_image.csv",
        index=False
    )

print(
    "Per-image results saved to:",
    OUTPUT_DIR
)


Per-image results saved to: /home/zahranadeem/thesis/cross_domain_benchmarks
